# Linear Regression with PySpark MLlib (2 CPU Cores)




## 1. Machine Learning in PySpark — Theory

PySpark's ML library is called **MLlib** (`pyspark.ml`). Unlike scikit-learn, it's built to train models on data **distributed across multiple cores/machines** rather than loaded into a single machine's memory.

### Key building blocks

| Component | What it does |
|---|---|
| **DataFrame** | Your dataset — like a distributed pandas DataFrame |
| **Transformer** | Takes a DataFrame, adds/changes columns (e.g., a *trained model* making predictions) |
| **Estimator** | An algorithm that is `.fit()` on data to produce a Transformer (e.g., `LinearRegression` → `LinearRegressionModel`) |

| **Evaluator** | Measures model performance (RMSE, R², etc.) |

### Typical workflow
1. Load data into a Spark DataFrame
2. Assemble feature columns into a single **vector column** using `VectorAssembler` (MLlib algorithms expect ONE features column, not many separate ones — this is different from scikit-learn's `X` matrix)
3. Split into train/test sets
4. Create the Estimator (e.g., `LinearRegression()`)
5. `.fit()` on training data → produces a Model (Transformer)
6. `.transform()` on test data → adds a `prediction` column
7. Evaluate using `RegressionEvaluator`

## 3. Hands-on: Step 1 — Install PySpark & Create a SparkSession (2 CPUs)

Note: for **MLlib** (DataFrame-based ML), we use `SparkSession` instead of the raw `SparkContext` used for RDDs — `SparkSession` is the modern entry point and internally manages a `SparkContext` for us.

In [1]:
# ---------------------------------------------------------
# STEP 1: Install PySpark in Colab
# ---------------------------------------------------------
!pip install pyspark -q   # -q keeps the install output quiet


In [2]:
# ---------------------------------------------------------
# STEP 2: Import SparkSession
# SparkSession is the modern entry point for DataFrame & ML APIs
# (it wraps SparkContext internally)
# ---------------------------------------------------------
from pyspark.sql import SparkSession

# ---------------------------------------------------------
# STEP 3: Create a SparkSession using 2 CPU cores locally
# "local[2]" -> run locally using 2 worker threads (2 CPUs)
# ---------------------------------------------------------
spark = SparkSession.builder \
    .appName("LinearRegression_using spark") \
    .master("local[2]") \
    .getOrCreate()

print("SparkSession created successfully!")
print("Running with:", spark.sparkContext.master)   # Should print local[2]


SparkSession created successfully!
Running with: local[2]


## 4. Hands-on: Step 2 — Create a Sample Dataset

We'll create a small synthetic dataset predicting **salary** from **years of experience** and **hours worked per week** — simple enough to see linear regression clearly, but the same steps apply to any real dataset (e.g., loaded via `spark.read.csv(...)`).

In [3]:
# ---------------------------------------------------------
# STEP 4: Create sample data as a list of tuples
# Columns: years_experience, hours_per_week, salary (target/label)
# ---------------------------------------------------------
data = [
    (1.0, 40.0, 32000.0),
    (2.0, 42.0, 38000.0),
    (3.0, 45.0, 45000.0),
    (4.0, 40.0, 50000.0),
    (5.0, 45.0, 58000.0),
    (6.0, 48.0, 62000.0),
    (7.0, 50.0, 70000.0),
    (8.0, 45.0, 75000.0),
    (9.0, 50.0, 82000.0),
    (10.0, 48.0, 90000.0),
    (11.0, 52.0, 95000.0),
    (12.0, 50.0, 100000.0),
]

columns = ["years_experience", "hours_per_week", "salary"]

# ---------------------------------------------------------
# STEP 5: Create a Spark DataFrame from the data
# ---------------------------------------------------------
df = spark.createDataFrame(data, columns)

# show() is like an action - it triggers computation and displays rows
df.show()
df.printSchema()


+----------------+--------------+--------+
|years_experience|hours_per_week|  salary|
+----------------+--------------+--------+
|             1.0|          40.0| 32000.0|
|             2.0|          42.0| 38000.0|
|             3.0|          45.0| 45000.0|
|             4.0|          40.0| 50000.0|
|             5.0|          45.0| 58000.0|
|             6.0|          48.0| 62000.0|
|             7.0|          50.0| 70000.0|
|             8.0|          45.0| 75000.0|
|             9.0|          50.0| 82000.0|
|            10.0|          48.0| 90000.0|
|            11.0|          52.0| 95000.0|
|            12.0|          50.0|100000.0|
+----------------+--------------+--------+

root
 |-- years_experience: double (nullable = true)
 |-- hours_per_week: double (nullable = true)
 |-- salary: double (nullable = true)



## 5. Hands-on: Step 3 — Feature Engineering with `VectorAssembler`

MLlib algorithms expect **one single "features" column** containing a vector of all input features — not separate columns like scikit-learn's `X`. `VectorAssembler` combines our input columns into that single vector column.

In [4]:
# ---------------------------------------------------------
# STEP 6: Import VectorAssembler
# This is a "Transformer" that combines multiple columns into
# one vector column called "features" (MLlib's required format)
# ---------------------------------------------------------
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["years_experience", "hours_per_week"],  # input feature columns
    outputCol="features"                                # combined output column
)

# transform() applies the assembler and adds the "features" column
df_features = assembler.transform(df)

# Now we have a "features" vector column + the original "salary" label column
df_features.select("features", "salary").show(truncate=False)


+-----------+--------+
|features   |salary  |
+-----------+--------+
|[1.0,40.0] |32000.0 |
|[2.0,42.0] |38000.0 |
|[3.0,45.0] |45000.0 |
|[4.0,40.0] |50000.0 |
|[5.0,45.0] |58000.0 |
|[6.0,48.0] |62000.0 |
|[7.0,50.0] |70000.0 |
|[8.0,45.0] |75000.0 |
|[9.0,50.0] |82000.0 |
|[10.0,48.0]|90000.0 |
|[11.0,52.0]|95000.0 |
|[12.0,50.0]|100000.0|
+-----------+--------+



## 6. Hands-on: Step 4 — Train/Test Split

In [5]:
# ---------------------------------------------------------
# STEP 7: Split data into training (80%) and testing (20%) sets
# randomSplit() divides the DataFrame randomly based on the given weights
# seed=42 makes the split reproducible
# ---------------------------------------------------------
train_data, test_data = df_features.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())


Training rows: 9
Testing rows: 3


## 7. Hands-on: Step 5 — Train the Linear Regression Model

This is the **Estimator → Model** pattern:
- `LinearRegression()` is the **Estimator** (untrained algorithm)
- Calling `.fit()` on training data produces a **Model**, which is a **Transformer**

In [6]:
# ---------------------------------------------------------
# STEP 8: Import and configure LinearRegression
# featuresCol -> the input vector column we created with VectorAssembler
# labelCol    -> the target/output column we want to predict
# ---------------------------------------------------------
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="features", labelCol="salary")

# ---------------------------------------------------------
# STEP 9: Train (fit) the model on the training data
# This is where the actual computation happens, distributed
# across our 2 CPU cores
# ---------------------------------------------------------
lr_model = lr.fit(train_data)

# Inspect the learned model parameters
print("Coefficients:", lr_model.coefficients)   # weight for each feature
print("Intercept:", lr_model.intercept)


Coefficients: [6375.120333725312,-27.917424323254153]
Intercept: 26271.04503154631


## 8. Hands-on: Step 6 — Make Predictions on Test Data

In [7]:
# ---------------------------------------------------------
# STEP 10: Use the trained model (a Transformer) to predict on test data
# transform() adds a new "prediction" column to the DataFrame
# ---------------------------------------------------------
predictions = lr_model.transform(test_data)

# Compare actual salary vs predicted salary
predictions.select("years_experience", "hours_per_week", "salary", "prediction").show()


+----------------+--------------+--------+------------------+
|years_experience|hours_per_week|  salary|        prediction|
+----------------+--------------+--------+------------------+
|             3.0|          45.0| 45000.0| 44140.12193817581|
|             7.0|          50.0| 70000.0| 69501.01615146079|
|            12.0|          50.0|100000.0|101376.61782008735|
+----------------+--------------+--------+------------------+



## 9. Hands-on: Step 7 — Evaluate the Model

We use `RegressionEvaluator` to measure how good our predictions are, using metrics like:
- **RMSE** (Root Mean Squared Error) — average prediction error, lower is better
- **R²** (R-squared) — how much variance in the target is explained by the model, closer to 1 is better

In [8]:
# ---------------------------------------------------------
# STEP 11: Evaluate the model using RegressionEvaluator
# ---------------------------------------------------------
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="salary", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="salary", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("RMSE:", rmse)
print("R2 Score:", r2)


RMSE: 980.3828818190681
R2 Score: 0.9980988229989726


## 12 Step 7 — Stop the SparkSession

In [10]:
# ---------------------------------------------------------
# STEP 15: Stop the SparkSession when done
# Releases the CPU threads/resources we were using
# ---------------------------------------------------------
spark.stop()
print("SparkSession stopped.")


SparkSession stopped.
